In [1]:
import gurobipy as GRB
from gurobipy import Model, GRB

In [2]:
# Create a new model
m = Model("capitalbudgeting")

# Create variables
x = [m.addVar(vtype=GRB.BINARY, name=f"x{i}") for i in range(1, 7)]

# Objective coefficients
NPV_adjusted = [9.5, 12.5, 12, 16, 18.5, 21]

# Set the objective
m.setObjective(sum(NPV_adjusted[i] * x[i] for i in range(6)), GRB.MAXIMIZE)

# Add constraint
m.addConstr(sum([5.5, 6.5, 7, 9, 10.5, 13][i] * x[i] for i in range(6)) <= 23, "c0")

# Optimize model
m.optimize()

# Print solution
for v in m.getVars():
    print('%s %g' % (v.varName, v.x))

print('Obj: %g' % m.objVal)

Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 24.6.0 24G517)

CPU model: Apple M4 Pro
Thread count: 14 physical cores, 14 logical processors, using up to 14 threads

Optimize a model with 1 rows, 6 columns and 6 nonzeros (Max)
Model fingerprint: 0x3d23ba4e
Model has 6 linear objective coefficients
Variable types: 0 continuous, 6 integer (6 binary)
Coefficient statistics:
  Matrix range     [6e+00, 1e+01]
  Objective range  [1e+01, 2e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+01, 2e+01]

Found heuristic solution: objective 34.0000000
Presolve time: 0.00s
Presolved: 1 rows, 6 columns, 6 nonzeros
Variable types: 0 continuous, 6 integer (6 binary)

Root relaxation: objective 4.111111e+01, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Ti

In [4]:
print("Optimal solution:", [v.x for v in m.getVars()])

Optimal solution: [1.0, -0.0, 1.0, -0.0, 1.0, -0.0]


## 常见 vtype（Gurobi）
- `GRB.BINARY`：二元变量（0/1）。
- `GRB.INTEGER`：整数变量（含正/负整数）。
- `GRB.CONTINUOUS`：连续变量（默认）。
- `GRB.SEMIINT`：半整数变量（要么 0，要么在区间 $[l,u]$ 的整数）。
- `GRB.SEMICONT`：半连续变量（要么 0，要么在区间 $[l,u]$ 的连续值）。

In [3]:
# How to iteratively check other optimal solutions for an IP
# Check if the model has an optimal solution
if m.status == GRB.OPTIMAL:
    # Store the optimal value
    optimal_value = m.objVal

    # List to store optimal solutions
    optimal_solutions = []

    # Function to add constraint to exclude the current solution
    def add_exclusion_constraint(m, solution):
        m.addConstr(sum(x[i] if solution[i] == 0 else (1-x[i]) for i in range(6)) >= 1)
        
        # solution[i] is the value of x[i] in the current optimal solution
        # The term equals 1 if x[i] changes
        # Equals 0 if x[i] stays the same
        # The sum counts how many variables differ from the current solution.

    # Loop to find all optimal solutions
    while True:
        # Store current solution
        current_solution = [var.X for var in x]
        optimal_solutions.append(current_solution)

        # Add constraint to exclude this solution
        add_exclusion_constraint(m, current_solution)

        # Re-optimize
        m.optimize()

        # Break if no more optimal solution is found
        if m.status != GRB.OPTIMAL or m.objVal < optimal_value:
            break

    # Print all optimal solutions
    for idx, solution in enumerate(optimal_solutions):
        print(f"Optimal Solution {idx + 1}: {solution} with Objective Value: {optimal_value}")
else:
    print("No optimal solution found.")


Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 24.6.0 24G517)

CPU model: Apple M4 Pro
Thread count: 14 physical cores, 14 logical processors, using up to 14 threads

Optimize a model with 2 rows, 6 columns and 12 nonzeros (Max)
Model fingerprint: 0x4bcd32ec
Model has 6 linear objective coefficients
Variable types: 0 continuous, 6 integer (6 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [1e+01, 2e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+00, 2e+01]

MIP start from previous solve did not produce a new incumbent solution
MIP start from previous solve violates constraint R1 by 1.000000000

Found heuristic solution: objective 34.0000000
Presolve time: 0.00s
Presolved: 2 rows, 6 columns, 11 nonzeros
Variable types: 0 continuous, 6 integer (6 binary)

Root relaxation: objective 4.111111e+01, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 E